# Telecom Retention Optimization: Phase 10 to 13 - Business Layer & Knapsack Optimization

This notebook covers:
1. Business Layer Setup ($V_i, c_i, s, E_i$)
2. 0/1 Knapsack Optimization Formulation (SciPy MILP & PuLP Solver)
3. Greedy Heuristic Comparison ($E_i / c_i$ ROI sorting)
4. Baseline Comparison (Random, Top-k Churn Prob, Top-k Value vs Optimized)
5. Sensitivity Analysis & Target List Export

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath('..'))
from src.model_trainer import ModelTrainer
from src.pipeline import build_preprocessing_pipeline
from src.business_optimizer import RetentionOptimizer
from src.baselines import StrategyEvaluator

## 1. Load Data, Model Predictions & Setup Business Layer

In [ ]:
df_clean = pd.read_csv(os.path.join('..', 'data', 'processed', 'processed_churn.csv'))
trainer = ModelTrainer(target_col='Churn', id_col='customerID')
X_train, X_test, y_train, y_test, ids_train, ids_test = trainer.split_data(df_clean)
preprocessor, _, _ = build_preprocessing_pipeline(df_clean, target_col='Churn', id_col='customerID')
model = trainer.train_logistic_regression(X_train, y_train, preprocessor)
churn_probs = model.predict_proba(X_test)[:, 1]

df_test_opt = X_test.copy()
df_test_opt['customerID'] = ids_test.values

optimizer = RetentionOptimizer(offer_success_rate=0.30, offer_cost_pct=0.15)
df_opt_results, summary = optimizer.optimize_retention(df_test_opt, churn_probs)
target_list = optimizer.export_target_list(df_opt_results, id_col='customerID', output_path=os.path.join('..', 'outputs', 'target_list.csv'))
target_list.head()

## 2. Baseline Comparison & Sensitivity Analysis

In [ ]:
plots_dir = os.path.join('..', 'outputs', 'plots')
strategy_eval = StrategyEvaluator(output_dir=plots_dir)
results_df = strategy_eval.evaluate_all_strategies(df_opt_results, budget=summary['budget'], y_test=y_test)
strategy_eval.plot_sensitivity_analysis(df_opt_results, base_budget=summary['budget'], optimizer_cls=RetentionOptimizer)